In [1]:
# Importe le module sys pour connaître l'interpréteur Python actuellement utilisé.
import sys
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# Importe Path pour afficher le dossier courant du notebook.
from pathlib import Path

# Importe boto3, le SDK Python permettant de communiquer avec AWS.
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs


import pandas as pd

In [2]:

s3 = fs.S3FileSystem(region="us-east-1")


# Indique le nom exact du bucket S3 contenant les données.
# Remplace cette valeur par le nom réel de ton bucket.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Équivalent AWS de PROJECT_ROOT.
# Dans notre cas, les fichiers Arrow sont dans le préfixe raw/.
PROJECT_ROOT = f"{BUCKET_NAME}/raw"

PARQUET_PATH = f"{PROJECT_ROOT}/parquet/mushrooms_merged.parquet"
PARQUET_OutPut = f"{BUCKET_NAME}/processed/parquet/mushrooms_CleanedNamesID.parquet"

table = pq.read_table(PARQUET_PATH,
    filesystem=s3)

print("Lignes :", table.num_rows)
print("Colonnes :", table.column_names)



Lignes : 2436
Colonnes : ['image', 'label', 'bbox', 'origin']


In [3]:
table.select(["label"]).slice(0, 10).to_pandas()

,label
0,Amanita_phalloides_poisonous
1,Amanita_phalloides_poisonous
2,Amanita_phalloides_poisonous
3,Amanita_phalloides_poisonous
4,Amanita_phalloides_poisonous
5,Amanita_phalloides_poisonous
6,Amanita_phalloides_poisonous
7,Amanita_phalloides_poisonous
8,Amanita_phalloides_poisonous
9,Amanita_phalloides_poisonous


In [4]:
labels = table["label"].to_pylist()

mushroom_names = []
edibilities = []

for label in labels:
    mushroom_name, edibility = label.rsplit("_", 1)
    # Convert the textual value into a Boolean.
    edibility = edibility == "edible"


    # Restaure le nom scientifique avec un espace.
    mushroom_name = mushroom_name.replace("_", " ")
    
    mushroom_names.append(mushroom_name)
    edibilities.append(edibility)

print("Mushroom tables label orig , name, edibility")

Mushroom tables label orig , name, edibility


In [5]:
pd.DataFrame({
    "label_original": labels[:10],
    "mushroom_name": mushroom_names[:10],
    "edibility": edibilities[:10]
})

table = table.append_column("mushroom_name", pa.array(mushroom_names))
table = table.append_column("edibility", pa.array(edibilities))

,label_original,mushroom_name,edibility
0,Amanita_phalloides_poisonous,Amanita phalloides,False
1,Amanita_phalloides_poisonous,Amanita phalloides,False
2,Amanita_phalloides_poisonous,Amanita phalloides,False
3,Amanita_phalloides_poisonous,Amanita phalloides,False
4,Amanita_phalloides_poisonous,Amanita phalloides,False
5,Amanita_phalloides_poisonous,Amanita phalloides,False
6,Amanita_phalloides_poisonous,Amanita phalloides,False
7,Amanita_phalloides_poisonous,Amanita phalloides,False
8,Amanita_phalloides_poisonous,Amanita phalloides,False
9,Amanita_phalloides_poisonous,Amanita phalloides,False


In [6]:
display(
    table.slice(0, 5).to_pandas()
)


entry_ids = [
    f"{index:06d}"
    for index in range(table.num_rows)
]

table = table.append_column(
    "entry_id",
    pa.array(entry_ids)
)

display(
    table.slice(0, 5).to_pandas()
)


,image,label,bbox,origin,mushroom_name,edibility
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[748.0, 285.0, 1011.75, 505.0]",natural,Amanita phalloides,False
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[178.0, 113.0, 445.5, 548.0]",natural,Amanita phalloides,False
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[471.0, 368.0, 594.75, 571.75]",natural,Amanita phalloides,False
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[553.0, 20.0, 854.25, 315.0]",natural,Amanita phalloides,False
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[484.0, 138.0, 672.75, 361.75]",natural,Amanita phalloides,False


,image,label,bbox,origin,mushroom_name,edibility,entry_id
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[748.0, 285.0, 1011.75, 505.0]",natural,Amanita phalloides,False,000000
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[178.0, 113.0, 445.5, 548.0]",natural,Amanita phalloides,False,000001
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[471.0, 368.0, 594.75, 571.75]",natural,Amanita phalloides,False,000002
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[553.0, 20.0, 854.25, 315.0]",natural,Amanita phalloides,False,000003
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[484.0, 138.0, 672.75, 361.75]",natural,Amanita phalloides,False,000004


In [7]:
pq.write_table(
    table,
    PARQUET_OutPut,
    filesystem=s3,
    compression="zstd"
)

print(PARQUET_OutPut)

mushroom-ml-mikael-2026-000/processed/parquet/mushrooms_CleanedNamesID.parquet
